In [86]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [87]:
model_name = "Qwen/Qwen3-0.6B"

In [88]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt},
    {"role": "assistant", "content": ""}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)

In [97]:
sample_thinking_trace = "ok so I'm thinking about the user's request. I'm going to give a short introduction to large language model that starts with"
prompt = "Give me a short introduction to large language model."

In [132]:
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}, {"role": "assistant", "content": ""}],
    add_generation_prompt=True,
    tokenize=False,
)
text = text.split("\n</think>\n")[0]

In [135]:
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    tokenize=False,
    enable_thinking=True
)

In [136]:
print(text)

<|im_start|>user
Give me a short introduction to large language model.<|im_end|>
<|im_start|>assistant



In [115]:
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

In [116]:
outputs = model.generate(**model_inputs, max_new_tokens=16000)

In [117]:
print(tokenizer.decode(outputs[0]))

<|im_start|>user
Give me a short introduction to large language model.<|im_end|>
<|im_start|>assistant
<think>
Okay, the user wants a short introduction to a large language model. Let me start by recalling what I know. Large language models are AI systems that can understand and generate text, right? They're trained on massive datasets to learn patterns and make predictions.

I should mention their ability to understand and generate human-like text. Maybe explain that they can work on various tasks like writing, answering questions, or even creating content. It's important to highlight their adaptability and efficiency. Also, perhaps touch on their applications in different fields. Let me check if I'm covering all key points without being too technical. Keep it concise but informative. Make sure the introduction flows well and is engaging. Avoid jargon but still sound knowledgeable. Alright, that should do it.
</think>

A large language model (LLM) is an AI system designed to understan

In [106]:
print(tokenizer.decode(outputs[0]))

<|im_start|>user
Give me a short introduction to large language model.<|im_end|>
<|im_start|>assistant
<think>
Okay, the user is asking for a short introduction to a large language model. Let me start by recalling what I know about LLMs. They're big language models, right? So I should mention their capabilities like understanding text, generating text, and handling various tasks.

I need to keep it concise. Maybe start with their general definition. Then explain what they do—like answering questions, writing content, etc. It's important to highlight their key features without getting too technical. Also, mention the training data and how they work. Oh, and maybe include some examples to make it relatable. Let me check if I'm covering all the essentials: definition, main functions, training, and maybe an example. Yeah, that should work. Keep the tone friendly and informative.
</think>

A large language model (LLM) is a type of artificial intelligence model designed to understand and gen

In [78]:
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, add_special_tokens=False).split("\n</think>\n")[0] + sample_thinking_trace
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

In [85]:
tokenizer.decode(tokenizer(text)['input_ids'])

"<|im_start|>user\nGive me a short introduction to large language model.<|im_end|>\n<|im_start|>assistant\n<think>\nok so I'm thinking about the user's request. I'm going to give a short introduction to large language model that starts with"

In [ ]:
outputs = model.generate(**model_inputs, max_new_tokens=16000)

In [67]:
print(tokenizer.decode(outputs[0]))

<|im_start|>user
Give me a short introduction to large language model.<|im_end|>
<|im_start|>assistant
<think>
ok so I'm thinking about the user's request. I'm going to give a short introduction to large language model that starts with a brief explanation of what it is, its purpose, and some key features. I should keep it concise and engaging, making sure to highlight the benefits and applications of such models.
</think>

A large language model (LLM) is a type of artificial intelligence designed to understand, generate, and respond to human language. These models are trained on vast amounts of text data to learn patterns and understand context, allowing them to perform a wide range of tasks, from answering questions to writing creative content. They are used in various applications such as customer service, content creation, and language translation.<|im_end|>


#### Annealing for Inference engine

- Generations for user to system tokens
- add in the annealing functions, which will anneal on tokens give on the inference side
- Input from the dataset that supports annleaing (dataloader side)
- instead of samples, reutrn back Prompt samples like a wrapper, at run time
- fix the logic up in train and inference
- make a scheduler for this lr, self.global_step which can be passed in

- how we parse the reward, look at the utils in the messages
- start experiments with the lr ablation for scheduling the annealing
- make sure all the pipelines work with this